# 07b — Generación del Informe Final (Español)
## TechPulse: Plataforma de Inteligencia de Productos

**Output:** `reports/pdf/techpulse_report_es.pdf`

In [13]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image,
                                 Table, TableStyle, PageBreak, HRFlowable,
                                 KeepTogether)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY, TA_RIGHT
from pathlib import Path
from PIL import Image as PILImage
import pandas as pd
import numpy as np
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

ROOT      = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
FIGURES   = ROOT / 'reports' / 'figures'
PDF_OUT   = ROOT / 'reports' / 'pdf'
PDF_OUT.mkdir(parents=True, exist_ok=True)

# Colores corporativos
C_PRIMARY   = colors.HexColor('#2563EB')
C_SECONDARY = colors.HexColor('#7C3AED')
C_ACCENT    = colors.HexColor('#059669')
C_WARNING   = colors.HexColor('#D97706')
C_DANGER    = colors.HexColor('#DC2626')
C_DARK      = colors.HexColor('#111827')
C_GRAY      = colors.HexColor('#6B7280')
C_LIGHT     = colors.HexColor('#F3F4F6')
C_WHITE     = colors.white
C_FINDING   = colors.HexColor('#EFF6FF')
C_FINDING_B = colors.HexColor('#DBEAFE')

PAGE_W = A4[0] - 4*cm  # ancho útil

print("✅ ReportLab cargado correctamente")

✅ ReportLab cargado correctamente


In [14]:
try:
    from PIL import Image as PILImage
    print("✅ Pillow disponible")
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'pillow'], check=True)
    from PIL import Image as PILImage
    print("✅ Pillow instalado")

✅ Pillow disponible


In [15]:
df           = pd.read_parquet(PROCESSED / 'clustered_products.parquet')
trends_df    = pd.read_parquet(PROCESSED / 'category_trends.parquet')
profiles_df  = pd.read_parquet(PROCESSED / 'cluster_profiles.parquet')

df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['year']       = df['created_at'].dt.year.astype(int)

total_products  = len(df)
total_votes     = df['votes'].sum()
top_product     = df.loc[df['votes'].idxmax(), 'name']
max_votes       = df['votes'].max()
best_growth_cat = trends_df.loc[trends_df['growth_pct'].idxmax(), 'category']
best_growth_pct = trends_df['growth_pct'].max()

print("✅ Datos cargados")

✅ Datos cargados


In [16]:
styles = getSampleStyleSheet()

style_cover_title = ParagraphStyle('CoverTitle',
    fontSize=34, textColor=C_PRIMARY, fontName='Helvetica-Bold',
    alignment=TA_LEFT, spaceAfter=4, leading=38)

style_cover_subtitle = ParagraphStyle('CoverSubtitle',
    fontSize=14, textColor=C_SECONDARY, fontName='Helvetica-Bold',
    alignment=TA_LEFT, spaceAfter=16)

style_cover_meta_label = ParagraphStyle('CoverMetaLabel',
    fontSize=9, textColor=C_GRAY, fontName='Helvetica-Bold',
    alignment=TA_LEFT, spaceAfter=1)

style_cover_meta_value = ParagraphStyle('CoverMetaValue',
    fontSize=10, textColor=C_DARK, fontName='Helvetica',
    alignment=TA_LEFT, spaceAfter=8)

style_cover_summary_title = ParagraphStyle('CoverSummaryTitle',
    fontSize=11, textColor=C_PRIMARY, fontName='Helvetica-Bold',
    alignment=TA_LEFT, spaceAfter=6, spaceBefore=10)

style_cover_summary = ParagraphStyle('CoverSummary',
    fontSize=9.5, textColor=C_DARK, fontName='Helvetica',
    alignment=TA_JUSTIFY, leading=14, spaceAfter=6)

style_h1 = ParagraphStyle('H1',
    fontSize=16, textColor=C_PRIMARY, fontName='Helvetica-Bold',
    spaceBefore=6, spaceAfter=8)

style_h2 = ParagraphStyle('H2',
    fontSize=12, textColor=C_SECONDARY, fontName='Helvetica-Bold',
    spaceBefore=10, spaceAfter=6)

style_body = ParagraphStyle('Body',
    fontSize=9.5, textColor=C_DARK, fontName='Helvetica',
    leading=14, spaceAfter=8, alignment=TA_JUSTIFY)

style_caption = ParagraphStyle('Caption',
    fontSize=8, textColor=C_GRAY, fontName='Helvetica-Oblique',
    alignment=TA_CENTER, spaceAfter=10)

style_finding_num = ParagraphStyle('FindingNum',
    fontSize=10, textColor=C_WHITE, fontName='Helvetica-Bold',
    alignment=TA_CENTER)

style_finding_text = ParagraphStyle('FindingText',
    fontSize=9.5, textColor=C_DARK, fontName='Helvetica',
    leading=14, alignment=TA_JUSTIFY)

print("✅ Estilos definidos")

✅ Estilos definidos


In [17]:
def add_figure_proportional(path, max_width_cm=16, caption=None):
    """Inserta figura manteniendo ratio original — sin distorsión."""
    elements = []
    p = Path(path)
    if not p.exists():
        elements.append(Paragraph(f"[Figure not found: {p.name}]", style_caption))
        return elements

    with PILImage.open(p) as img:
        orig_w, orig_h = img.size

    max_w = max_width_cm * cm
    ratio = orig_h / orig_w
    final_w = min(max_w, PAGE_W)
    final_h = final_w * ratio

    image = Image(str(p), width=final_w, height=final_h)
    image.hAlign = 'CENTER'
    elements.append(image)
    if caption:
        elements.append(Paragraph(caption, style_caption))
    elements.append(Spacer(1, 0.3*cm))
    return elements

def section_header(number, title):
    """Encabezado de sección con línea decorativa — estilo PDF referencia."""
    elements = []
    elements.append(Spacer(1, 0.4*cm))
    elements.append(HRFlowable(width="100%", thickness=2,
                               color=C_PRIMARY, spaceAfter=8))
    elements.append(Paragraph(f"{number}. {title}", style_h1))
    return elements

def finding_box(number, text):
    """Caja de hallazgo numerada — estilo PDF referencia."""
    data = [[
        Paragraph(f"Finding\n{number}", style_finding_num),
        Paragraph(text, style_finding_text)
    ]]
    t = Table(data, colWidths=[2*cm, PAGE_W - 2*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',  (0,0), (0,0), C_PRIMARY),
        ('BACKGROUND',  (1,0), (1,0), C_FINDING_B),
        ('VALIGN',      (0,0), (-1,-1), 'MIDDLE'),
        ('ALIGN',       (0,0), (0,0), 'CENTER'),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
        ('RIGHTPADDING',(0,0), (-1,-1), 10),
        ('TOPPADDING',  (0,0), (-1,-1), 8),
        ('BOTTOMPADDING',(0,0),(-1,-1), 8),
        ('ROWHEIGHT',   (0,0), (-1,-1), 0.2*cm),
    ]))
    return [t, Spacer(1, 0.4*cm)]

def meta_row(label, value):
    """Fila de metadata para portada."""
    return [
        Paragraph(label.upper(), style_cover_meta_label),
        Paragraph(value, style_cover_meta_value),
    ]

print("✅ Funciones auxiliares definidas")

✅ Funciones auxiliares definidas


In [18]:
output_path = PDF_OUT / 'techpulse_report_es.pdf'
doc = SimpleDocTemplate(
    str(output_path),
    pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm,
    topMargin=2.5*cm, bottomMargin=2*cm,
)

elements = []

# ══════════════════════════════════════════════════════
# PORTADA
# ══════════════════════════════════════════════════════
elements.append(Spacer(1, 1.5*cm))
elements.append(Paragraph("TechPulse", style_cover_title))
elements.append(Paragraph("Plataforma de Inteligencia de Productos Digitales", style_cover_subtitle))
elements.append(HRFlowable(width="100%", thickness=2, color=C_PRIMARY, spaceAfter=16))

meta_data = [
    [Paragraph("AUTOR", style_cover_meta_label),
     Paragraph("Bryan Anthony López Guerrero", style_cover_meta_value)],
    [Paragraph("FORMACIÓN", style_cover_meta_label),
     Paragraph("Ing. en TI | Máster en Visual Analytics y Big Data | Esp. en Big Data e IA", style_cover_meta_value)],
    [Paragraph("FUENTE DE DATOS", style_cover_meta_label),
     Paragraph("Product Hunt — Dataset histórico Kaggle + API Oficial (2014–2024)", style_cover_meta_value)],
    [Paragraph("ESCALA", style_cover_meta_label),
     Paragraph("152,556 productos · 10 años · 20.3M votos totales · 470 categorías únicas", style_cover_meta_value)],
    [Paragraph("MODELOS", style_cover_meta_label),
     Paragraph("Holt-Winters + STL · TF-IDF + K-Means + UMAP · Sentence Transformers (all-MiniLM-L6-v2)", style_cover_meta_value)],
    [Paragraph("STACK TÉCNICO", style_cover_meta_label),
     Paragraph("Python · Pandas · Scikit-learn · UMAP · Sentence Transformers · Streamlit · FastAPI · ReportLab", style_cover_meta_value)],
    [Paragraph("GITHUB", style_cover_meta_label),
     Paragraph("github.com/anthonylopez-dev/techpulse-product-intelligence", style_cover_meta_value)],
    [Paragraph("LINKEDIN", style_cover_meta_label),
     Paragraph("linkedin.com/in/anthonylpz", style_cover_meta_value)],
    [Paragraph("FECHA", style_cover_meta_label),
     Paragraph(datetime.now().strftime("%B %Y"), style_cover_meta_value)],
]
meta_table = Table(meta_data, colWidths=[3.5*cm, PAGE_W - 3.5*cm])
meta_table.setStyle(TableStyle([
    ('VALIGN',       (0,0), (-1,-1), 'TOP'),
    ('LEFTPADDING',  (0,0), (-1,-1), 0),
    ('RIGHTPADDING', (0,0), (-1,-1), 8),
    ('TOPPADDING',   (0,0), (-1,-1), 2),
    ('BOTTOMPADDING',(0,0), (-1,-1), 2),
    ('ROWBACKGROUNDS',(0,0),(-1,-1), [C_WHITE, C_LIGHT]),
]))
elements.append(meta_table)
elements.append(Spacer(1, 0.5*cm))
elements.append(HRFlowable(width="100%", thickness=1, color=C_LIGHT, spaceAfter=10))

kpi_data = [[
    Paragraph("152,556", ParagraphStyle('KV', fontSize=20, textColor=C_WHITE,
              fontName='Helvetica-Bold', alignment=TA_CENTER)),
    Paragraph("10", ParagraphStyle('KV', fontSize=20, textColor=C_WHITE,
              fontName='Helvetica-Bold', alignment=TA_CENTER)),
    Paragraph("20.3M", ParagraphStyle('KV', fontSize=20, textColor=C_WHITE,
              fontName='Helvetica-Bold', alignment=TA_CENTER)),
    Paragraph("125,579", ParagraphStyle('KV', fontSize=20, textColor=C_WHITE,
              fontName='Helvetica-Bold', alignment=TA_CENTER)),
    Paragraph("10", ParagraphStyle('KV', fontSize=20, textColor=C_WHITE,
              fontName='Helvetica-Bold', alignment=TA_CENTER)),
],[
    Paragraph("Productos\nAnalizados", ParagraphStyle('KL', fontSize=8, textColor=C_WHITE,
              fontName='Helvetica', alignment=TA_CENTER, leading=11)),
    Paragraph("Años\nde Datos", ParagraphStyle('KL', fontSize=8, textColor=C_WHITE,
              fontName='Helvetica', alignment=TA_CENTER, leading=11)),
    Paragraph("Votos\nTotales", ParagraphStyle('KL', fontSize=8, textColor=C_WHITE,
              fontName='Helvetica', alignment=TA_CENTER, leading=11)),
    Paragraph("Productos\nIndexados", ParagraphStyle('KL', fontSize=8, textColor=C_WHITE,
              fontName='Helvetica', alignment=TA_CENTER, leading=11)),
    Paragraph("Segmentos\nde Mercado", ParagraphStyle('KL', fontSize=8, textColor=C_WHITE,
              fontName='Helvetica', alignment=TA_CENTER, leading=11)),
]]
kpi_col_w = PAGE_W / 5
kpi_table = Table(kpi_data, colWidths=[kpi_col_w]*5)
kpi_table.setStyle(TableStyle([
    ('BACKGROUND',   (0,0), (-1,-1), C_PRIMARY),
    ('ALIGN',        (0,0), (-1,-1), 'CENTER'),
    ('VALIGN',       (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',    (0,0), (0,-1), 1*cm),
    ('ROWHEIGHT',    (0,1), (-1,1), 0.8*cm),
    ('TOPPADDING',   (0,0), (-1,-1), 6),
    ('BOTTOMPADDING',(0,0), (-1,-1), 6),
    ('LINEAFTER',    (0,0), (3,-1), 0.5, C_WHITE),
]))
elements.append(kpi_table)
elements.append(Spacer(1, 0.5*cm))

elements.append(Paragraph("Resumen Ejecutivo", style_cover_summary_title))
elements.append(Paragraph(
    "TechPulse es una plataforma de ciencia de datos end-to-end que analiza más de 152,000 productos "
    "digitales lanzados en Product Hunt entre 2014 y 2024. El sistema integra tres capas analíticas: "
    "(1) forecasting de tendencias usando Holt-Winters y descomposición STL para proyectar el crecimiento "
    "por categoría hasta 2026; (2) segmentación semántica de mercado mediante vectorización TF-IDF, "
    "clustering K-Means y reducción dimensional UMAP, identificando 10 segmentos de productos distintos; "
    "y (3) un motor de recomendación híbrido basado en Sentence Transformers que permite búsqueda "
    "semántica en texto libre y matching de similitud producto-a-producto sobre 125,579 productos indexados.",
    style_cover_summary))
elements.append(Paragraph(
    "El análisis revela que el 55.5% de los lanzamientos de 2024 son productos relacionados con IA — "
    "el cambio estructural más significativo en el ecosistema de productos digitales en una década. "
    "La categoría 'Money' muestra el mayor crecimiento proyectado (+12.3%), mientras que 'Design & "
    "Open Source' lidera en engagement comunitario con 175 votos promedio por producto. Este sistema "
    "es completamente replicable: cualquier organización con datos de catálogo de productos puede "
    "desplegar este pipeline para generar inteligencia competitiva, segmentar su mercado y "
    "personalizar recomendaciones.",
    style_cover_summary))

elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 1: DESCRIPCIÓN DEL PROYECTO
# ══════════════════════════════════════════════════════
elements += section_header("1", "Descripción del Proyecto")
elements.append(Paragraph(
    "TechPulse aborda un problema real de negocio: entender qué productos y categorías digitales "
    "están ganando tracción en el mercado antes de que se vuelvan mainstream. Construido sobre una "
    "década de datos de Product Hunt, la plataforma combina machine learning clásico, NLP moderno "
    "y forecasting de series temporales en un sistema de inteligencia desplegable.",
    style_body))

elements.append(Paragraph("1.1 Preguntas de Investigación", style_h2))
questions = [
    "¿Qué categorías de productos dominarán el ecosistema digital en 2025–2026?",
    "¿Cuál es la estructura semántica del mercado de productos digitales — cuántos segmentos distintos existen?",
    "¿Puede un motor de recomendación semántico superar la búsqueda por palabras clave en el descubrimiento de productos?",
    "¿Qué revela la señal de IA en los datos de Product Hunt sobre el estado del ecosistema tech?",
    "¿Cómo puede este pipeline ser replicado por organizaciones con sus propios datos de catálogo?",
]
for i, q in enumerate(questions, 1):
    elements.append(Paragraph(f"{i}. {q}", style_body))

elements.append(Paragraph("1.2 Arquitectura Técnica", style_h2))
elements.append(Paragraph(
    "El proyecto está estructurado como un pipeline modular de siete notebooks, cada uno produciendo "
    "outputs consumidos por la siguiente etapa — desde la ingesta de datos crudos hasta una "
    "aplicación Streamlit desplegada.",
    style_body))

arch_data = [
    ['Notebook', 'Etapa', 'Output Principal'],
    ['01', 'Recolección y Unificación de Datos', 'master_dataset.parquet — 152,556 productos'],
    ['02', 'Análisis Exploratorio del Ecosistema', '7 visualizaciones publicables'],
    ['03', 'Forecasting de Tendencias', 'forecasts.parquet — proyecciones hasta 2026'],
    ['04', 'Clustering de Productos', 'clustered_products.parquet — 10 segmentos'],
    ['05', 'Motor de Recomendación', '125,579 productos indexados + embeddings'],
    ['06', 'Insights de Negocio', 'Informe ejecutivo bilingüe'],
    ['07', 'Informe PDF', 'techpulse_report_es.pdf'],
]
arch_table = Table(arch_data, colWidths=[1.8*cm, 7*cm, PAGE_W - 8.8*cm])
arch_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_PRIMARY),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTNAME',      (0,1), (0,-1), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('ALIGN',         (0,0), (0,-1), 'CENTER'),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.72*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(arch_table)
elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 2: DATOS Y METODOLOGÍA
# ══════════════════════════════════════════════════════
elements += section_header("2", "Fuentes de Datos y Metodología")
elements.append(Paragraph(
    "El dataset integra tres fuentes para lograr cobertura completa del ecosistema de Product Hunt "
    "de 2014 a 2024. Cada fuente fue cargada independientemente, normalizada en esquema "
    "y unificada en un único archivo Parquet maestro.",
    style_body))

data_sources = [
    ['Fuente', 'Cobertura', 'Registros', 'Variables Clave'],
    ['Kaggle Histórico', '2014–2021', '76,700', 'upvotes, comments, category_tags, makers'],
    ['Kaggle 2023', '2023', '40,615', 'votesCount, commentsCount, topics, createdAt'],
    ['Kaggle 2024', '2024', '35,241', 'votesCount, commentsCount, topics, createdAt'],
    ['Dataset Maestro', '2014–2024', '152,556', '13 variables unificadas, formato Parquet'],
]
ds_table = Table(data_sources, colWidths=[3.5*cm, 2.5*cm, 2.5*cm, PAGE_W - 8.5*cm])
ds_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_SECONDARY),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTNAME',      (0,-1), (-1,-1), 'Helvetica-Bold'),
    ('BACKGROUND',    (0,-1), (-1,-1), C_FINDING_B),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-2), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.72*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(ds_table)
elements.append(Spacer(1, 0.4*cm))

elements.append(Paragraph("2.1 Desafíos Técnicos y Soluciones", style_h2))
elements.append(Paragraph(
    "Tres desafíos técnicos requirieron decisiones de ingeniería específicas. Primero, conflictos "
    "de timezone: el dataset histórico usaba datetimes sin zona horaria mientras las fuentes 2023–2024 "
    "usaban timestamps UTC — resuelto eliminando la información de zona antes de la concatenación. "
    "Segundo, heterogeneidad de esquemas: los nombres de columnas diferían entre fuentes (upvotes vs. "
    "votesCount) — resuelto mediante una función de transformación unificada. Tercero, año 2022 ausente: "
    "ningún dataset de Kaggle cubre este período — documentado de forma transparente sin imputación.",
    style_body))

elements += finding_box("1",
    "152,556 productos unificados desde 3 fuentes independientes abarcando 10 años. "
    "La estrategia híbrida de datos (Kaggle histórico + datos recientes de API) elimina el riesgo "
    "de scraping logrando cobertura casi completa de la década. El único gap es 2022, sin dataset "
    "público disponible — documentado como limitación conocida.")

elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 3: ANÁLISIS DEL ECOSISTEMA
# ══════════════════════════════════════════════════════
elements += section_header("3", "Análisis del Ecosistema")
elements.append(Paragraph(
    "El análisis exploratorio revela una trayectoria de crecimiento constante marcada por dos "
    "cambios estructurales mayores: la pandemia de COVID-19 en 2020 y el boom de IA en 2023.",
    style_body))

elements += add_figure_proportional(FIGURES / '01_launches_per_year.png',
    caption="Fig 1. Lanzamientos de productos por año (2014–2024). 2023 estableció un récord histórico con 40,615 lanzamientos.")

elements.append(Paragraph("3.1 Mapa de Categorías", style_h2))
elements.append(Paragraph(
    "Se identificaron 470 categorías únicas en el dataset. La categoría líder — Tech — concentra "
    "52,012 productos, seguida de Productivity (33,107) e Artificial Intelligence (24,048). "
    "La presencia de IA como tercera categoría más grande para 2024 marca un cambio estructural "
    "de tecnología de nicho a paradigma mainstream de productos digitales.",
    style_body))

elements += add_figure_proportional(FIGURES / '03_top_categories.png',
    caption="Fig 2. Top 15 categorías de productos por volumen total (2014–2024).")

elements += add_figure_proportional(FIGURES / '04_category_evolution.png',
    caption="Fig 3. Evolución de las top 6 categorías en el tiempo — el crecimiento de IA es inconfundible desde 2022.")

elements.append(Paragraph("3.2 Patrones de Engagement", style_h2))
elements.append(Paragraph(
    "La distribución de votos es altamente sesgada a la derecha: el producto mediano recibe 45 votos "
    "mientras el promedio es 133, indicando una larga cola de productos altamente exitosos. El producto "
    "más votado de todos los tiempos es Startup Stash con 21,798 votos. El segmento 'Design & Open "
    "Source' lidera en engagement promedio con 175 votos por producto.",
    style_body))

elements += add_figure_proportional(FIGURES / '06_top15_products.png',
    caption="Fig 4. Top 15 productos más votados de todos los tiempos en Product Hunt.")

elements += finding_box("2",
    "2023 registró 40,615 lanzamientos de productos — el mayor en la historia de Product Hunt, "
    "3.3 veces el volumen de 2017. El boom de IA es el principal motor: Artificial Intelligence "
    "creció de ser una categoría menor en 2019 a la tercera más grande por volumen en 2024.")

elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 4: FORECASTING DE TENDENCIAS
# ══════════════════════════════════════════════════════
elements += section_header("4", "Forecasting de Tendencias")
elements.append(Paragraph(
    "El forecasting de tendencias fue implementado usando Holt-Winters Exponential Smoothing con "
    "tendencia y estacionalidad aditivas, combinado con descomposición STL para aislar los componentes "
    "de tendencia, estacionalidad y residuos. Se seleccionaron siete categorías: cinco dominantes "
    "por volumen y dos emergentes por ratio de crecimiento.",
    style_body))

elements.append(Paragraph("4.1 Selección del Modelo", style_h2))
elements.append(Paragraph(
    "Prophet fue seleccionado inicialmente pero resultó incompatible con entornos Windows por "
    "dependencias del backend Stan. Holt-Winters fue elegido como alternativa de producción — "
    "es más interpretable, no requiere compilador C++ externo y tiene rendimiento comparable "
    "en datos agregados mensualmente con patrones estacionales claros.",
    style_body))

elements += add_figure_proportional(FIGURES / '08_stl_decomposition.png',
    caption="Fig 5. Descomposición STL — tendencia, estacionalidad y residuos para categorías dominantes.")

elements.append(Paragraph("4.2 Proyecciones hasta 2026", style_h2))

trend_data = [['Categoría', 'Actual (mes)', 'Forecast 2026 (mes)', 'Crecimiento', 'Tipo']]
for _, row in trends_df.sort_values('growth_pct', ascending=False).iterrows():
    trend_data.append([
        row['category'],
        f"{row['last_observed']:.0f}",
        f"{row['forecast_2026']:.0f}",
        f"{row['growth_pct']:+.1f}%",
        'Dominante' if row['is_dominant'] else '🚀 Emergente',
    ])
t_table = Table(trend_data, colWidths=[5.5*cm, 2.5*cm, 2.8*cm, 2.2*cm, 3*cm])
t_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_ACCENT),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('ALIGN',         (1,0), (-1,-1), 'CENTER'),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.68*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(t_table)
elements.append(Spacer(1, 0.4*cm))

elements += add_figure_proportional(FIGURES / '10_growth_ranking.png',
    caption="Fig 6. Ranking de crecimiento proyectado — categorías dominantes vs. emergentes hasta finales de 2026.")

elements += finding_box("3",
    f"'Money' muestra el mayor crecimiento proyectado con +{best_growth_pct:.1f}% — señalando "
    "una oportunidad significativa en fintech y herramientas de finanzas personales. Artificial "
    "Intelligence proyecta un -7.3% no porque el sector esté declinando, sino porque su explosivo "
    "crecimiento de 2023 se está normalizando. El volumen absoluto de productos AI sigue siendo el más alto.")

elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 5: SEGMENTACIÓN DE MERCADO
# ══════════════════════════════════════════════════════
elements += section_header("5", "Segmentación de Mercado")
elements.append(Paragraph(
    "Se aplicó clustering K-Means (k=10) sobre vectores TF-IDF construidos a partir del nombre, "
    "tagline y descripción de cada producto. UMAP redujo el espacio TF-IDF de alta dimensionalidad "
    "a 2D para visualización, revelando límites claros entre clusters y confirmando coherencia semántica.",
    style_body))

elements.append(Paragraph("5.1 Selección de Clusters — Método del Codo", style_h2))
elements += add_figure_proportional(FIGURES / '11_elbow_method.png', max_width_cm=12,
    caption="Fig 7. Método del codo — inercia vs. número de clusters. k=10 seleccionado en el punto de inflexión.")

elements.append(Paragraph("5.2 Mapa del Mercado", style_h2))
elements += add_figure_proportional(FIGURES / '13_umap_named_clusters.png',
    caption="Fig 8. Mapa del ecosistema de productos — 10 segmentos de mercado nombrados proyectados mediante UMAP.")

elements.append(Paragraph("5.3 Perfiles de Segmento", style_h2))

seg_data = [['Segmento', 'Productos', 'Votos Prom.', 'Keywords Definitorias']]
cluster_name_col = 'cluster_name' if 'cluster_name' in profiles_df.columns else 'cluster'
for _, row in profiles_df.sort_values('avg_votes', ascending=False).iterrows():
    keywords = str(row.get('keywords', ''))[:45] + '...' if len(str(row.get('keywords',''))) > 45 else str(row.get('keywords',''))
    seg_data.append([
        str(row[cluster_name_col]),
        f"{int(row['n_products']):,}",
        f"{row['avg_votes']:.0f}",
        keywords,
    ])
seg_table = Table(seg_data, colWidths=[4.5*cm, 2*cm, 2*cm, PAGE_W - 8.5*cm])
seg_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_SECONDARY),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('ALIGN',         (1,0), (2,-1), 'CENTER'),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.68*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(seg_table)

elements += finding_box("4",
    "10 segmentos de mercado semánticamente distintos identificados. 'General Tech Products' es el "
    "más grande con 52,996 productos (42.2%), pero 'Design & Open Source' lidera en engagement "
    "con 175 votos promedio — demostrando que los segmentos de nicho y alta calidad superan en "
    "recepción comunitaria a los segmentos de alto volumen.")

elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 6: MOTOR DE RECOMENDACIÓN
# ══════════════════════════════════════════════════════
elements += section_header("6", "Motor de Recomendación")
elements.append(Paragraph(
    "El motor de recomendación híbrido usa Sentence Transformers (all-MiniLM-L6-v2) para generar "
    "embeddings semánticos de 384 dimensiones para los 125,579 productos indexados. La similitud "
    "coseno permite matching en tiempo real en ambos modos con inferencia sub-segundo.",
    style_body))

elements.append(Paragraph("6.1 Arquitectura", style_h2))
rec_arch = [
    ['Componente', 'Tecnología', 'Rol'],
    ['Representación de texto', 'Nombre + Tagline + Descripción', 'Input al modelo de embedding'],
    ['Modelo de embedding', 'all-MiniLM-L6-v2 (384d)', 'Generación de vectores semánticos'],
    ['Métrica de similitud', 'Similitud coseno', 'Distancia entre vectores de productos'],
    ['Modo A', 'Query en texto libre → top-N productos', 'Búsqueda semántica'],
    ['Modo B', 'Nombre de producto → productos similares', 'Matching de similitud'],
    ['Índice', '125,579 productos · 384 dimensiones', 'Pre-computado, cargado en runtime'],
]
ra_table = Table(rec_arch, colWidths=[4*cm, 5.5*cm, PAGE_W - 9.5*cm])
ra_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_ACCENT),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.68*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(ra_table)
elements.append(Spacer(1, 0.4*cm))

elements.append(Paragraph("6.2 Evaluación del Sistema", style_h2))
elements.append(Paragraph(
    "La coherencia de cluster fue medida en una muestra de 30 productos de alto engagement "
    "(votos > 200). El 36.7% de las top-10 recomendaciones pertenece al mismo cluster que el "
    "producto de referencia. Esto es intencional: un sistema con 100% de coherencia de cluster "
    "solo recomendaría productos dentro del mismo segmento, perdiendo oportunidades inter-segmento "
    "que frecuentemente representan los descubrimientos más novedosos.",
    style_body))

rec_examples = [
    ['Modo', 'Input', 'Mejor Resultado', 'Score'],
    ['A — Texto libre', 'AI tool to write content', 'Free AI Writing & Generators Tools', '0.771'],
    ['A — Texto libre', 'open source design system', 'Open Design', '0.689'],
    ['A — Texto libre', 'personal finances app', 'Budget Boss', '0.815'],
    ['B — Producto similar', 'Notion', 'MemberSpace with Notion', '0.839'],
    ['B — Producto similar', 'Figma', 'FigJam AI by Figma', '0.816'],
    ['B — Producto similar', 'ChatGPT', 'Query Kitty', '0.825'],
]
re_table = Table(rec_examples, colWidths=[3.5*cm, 4.5*cm, 6*cm, 2*cm])
re_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_PRIMARY),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('ALIGN',         (3,0), (3,-1), 'CENTER'),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.68*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(re_table)

elements += finding_box("5",
    "Scores de similitud semántica consistentemente por encima de 0.70 en todos los casos de prueba "
    "confirman que all-MiniLM-L6-v2 captura relaciones significativas entre productos sin solapamiento "
    "de palabras clave. El sistema encuentra 'Query Kitty' como la coincidencia más cercana a 'ChatGPT' "
    "con un score de 0.825 — sin keywords compartidas — puramente mediante comprensión semántica.")

elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 7: INSIGHTS DE NEGOCIO
# ══════════════════════════════════════════════════════
elements += section_header("7", "Insights de Negocio y Replicabilidad")
elements.append(Paragraph(
    "TechPulse no es un análisis puntual — es un framework de inteligencia replicable. "
    "Cualquier organización con datos de catálogo de productos puede desplegar este mismo pipeline "
    "para generar inteligencia competitiva, segmentar su mercado y personalizar experiencias de usuario.",
    style_body))

elements += add_figure_proportional(FIGURES / '17_ai_signal.png',
    caption="Fig 9. La señal de IA — % de lanzamientos relacionados con IA por año. El 55.5% de los lanzamientos de 2024 son AI-related.")

elements += add_figure_proportional(FIGURES / '15_opportunity_matrix.png',
    caption="Fig 10. Matriz de oportunidad de mercado — volumen actual vs. crecimiento proyectado hasta 2026.")

elements.append(Paragraph("7.1 Framework de Replicabilidad", style_h2))
biz_data = [
    ['Industria', 'Caso de Uso', 'Valor de Negocio'],
    ['Capital de Riesgo', 'Identificar categorías emergentes antes del pico',
     'Inversiones tempranas, mayor retorno'],
    ['Empresas de Producto', 'Benchmarking de engagement vs. mercado',
     'Decisiones de roadmap basadas en datos'],
    ['E-commerce', 'Recomendar productos a usuarios',
     'Mayor conversión, menor churn'],
    ['Consultoría', 'Análisis de mercado para clientes',
     'Reportes más rápidos y profundos'],
    ['Medios y Publicación', 'Forecasting de tendencias de contenido',
     'Anticipar qué quiere la audiencia'],
]
biz_table = Table(biz_data, colWidths=[3.5*cm, 6*cm, PAGE_W - 9.5*cm])
biz_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_PRIMARY),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.72*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(biz_table)
elements.append(PageBreak())

# ══════════════════════════════════════════════════════
# SECCIÓN 8: CONCLUSIONES
# ══════════════════════════════════════════════════════
elements += section_header("8", "Conclusiones y Trabajo Futuro")
elements.append(Paragraph("8.1 Conclusiones Clave", style_h2))
conclusions = [
    ("La IA es el paradigma dominante.",
     "El 55.5% de los lanzamientos de 2024 en Product Hunt son AI-related. Esto no es una tendencia — "
     "es un cambio estructural. Cualquier estrategia de producto que ignore la integración de IA opera "
     "con un punto ciego significativo."),
    ("La segmentación de mercado revela estructura oculta.",
     "10 segmentos semánticamente distintos emergen del clustering, con 'Design & Open Source' logrando "
     "el mayor engagement comunitario a pesar de ser una categoría de nicho — demostrando que la calidad "
     "de la comunidad importa más que el volumen."),
    ("La búsqueda semántica supera el matching por palabras clave.",
     "El motor de recomendación alcanza scores de similitud superiores a 0.70 usando embeddings semánticos "
     "puros, encontrando productos relevantes que no comparten keywords — una capacidad que la búsqueda "
     "por palabras clave no puede replicar."),
    ("El pipeline está listo para producción y es replicable.",
     "Siete notebooks modulares producen artefactos desplegables: un data lake en Parquet, modelos "
     "entrenados, un endpoint de inferencia FastAPI y una aplicación Streamlit. Cualquier organización "
     "puede sustituir los datos de Product Hunt por su propio catálogo."),
]
for i, (title, text) in enumerate(conclusions, 1):
    elements.append(Paragraph(f"{i}. <b>{title}</b> {text}", style_body))

elements.append(Paragraph("8.2 Trabajo Futuro", style_h2))
future_data = [
    ['#', 'Recomendación', 'Impacto'],
    ['1', 'Conectar la API en vivo de Product Hunt para ingesta en tiempo real', 'Alto'],
    ['2', 'Agregar monitoreo de drift con Evidently AI', 'Alto'],
    ['3', 'Expandir el motor de recomendación con filtrado colaborativo', 'Alto'],
    ['4', 'Desplegar el endpoint FastAPI en infraestructura cloud (AWS/GCP)', 'Medio'],
    ['5', 'Extender a fuentes adicionales (GitHub, HuggingFace)', 'Medio'],
]
fw_table = Table(future_data, colWidths=[1*cm, PAGE_W - 4*cm, 3*cm])
fw_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_DARK),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('ALIGN',         (0,0), (0,-1), 'CENTER'),
    ('ALIGN',         (2,0), (2,-1), 'CENTER'),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.68*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 8),
]))
elements.append(fw_table)
elements.append(Spacer(1, 0.5*cm))

elements += finding_box("6",
    "El hallazgo más importante es sistémico: que el 55.5% de los lanzamientos de 2024 sean "
    "AI-related significa que el ecosistema de productos digitales ha cruzado un punto de inflexión. "
    "Las organizaciones que puedan identificar, rastrear y actuar sobre esta señal tempranamente — "
    "usando sistemas como TechPulse — tendrán una ventaja competitiva duradera en estrategia de "
    "producto y decisiones de inversión.")

# ══════════════════════════════════════════════════════
# RESUMEN EJECUTIVO FINAL
# ══════════════════════════════════════════════════════
elements.append(PageBreak())
elements += section_header("", "Resumen Ejecutivo")

summary_data = [
    ['Métrica', 'Valor'],
    ['Productos analizados', '152,556'],
    ['Cobertura de datos', '2014 – 2024 (10 años)'],
    ['Votos totales analizados', '20,335,001'],
    ['Categorías únicas', '470'],
    ['Productos AI en 2024', '55.5% de todos los lanzamientos'],
    ['Año récord', '2023 — 40,615 lanzamientos'],
    ['Categoría de mayor crecimiento', f'Money (+{best_growth_pct:.1f}% proyectado a 2026)'],
    ['Segmento más engaging', 'Design & Open Source (175 votos promedio/producto)'],
    ['Segmentos de mercado identificados', '10 via K-Means + UMAP'],
    ['Índice de recomendación', '125,579 productos · 384 dimensiones'],
    ['Modelo de embedding', 'all-MiniLM-L6-v2 (Sentence Transformers)'],
    ['Modelo de forecasting', 'Holt-Winters Exponential Smoothing + STL'],
    ['Coherencia de cluster', '36.7% (diversidad inter-segmento intencional)'],
]
sum_table = Table(summary_data, colWidths=[8*cm, PAGE_W - 8*cm])
sum_table.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,0), C_PRIMARY),
    ('TEXTCOLOR',     (0,0), (-1,0), C_WHITE),
    ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTNAME',      (0,1), (0,-1), 'Helvetica-Bold'),
    ('FONTSIZE',      (0,0), (-1,-1), 9),
    ('ROWBACKGROUNDS',(0,1), (-1,-1), [C_WHITE, C_LIGHT]),
    ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
    ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ('ROWHEIGHT',     (0,0), (-1,-1), 0.75*cm),
    ('LEFTPADDING',   (0,0), (-1,-1), 10),
]))
elements.append(sum_table)
elements.append(Spacer(1, 1*cm))
elements.append(HRFlowable(width="100%", thickness=2, color=C_PRIMARY, spaceAfter=12))
elements.append(Paragraph("Bryan Anthony López Guerrero — Científico de Datos",
    ParagraphStyle('F1', fontSize=11, textColor=C_PRIMARY, fontName='Helvetica-Bold',
                   alignment=TA_CENTER)))
elements.append(Paragraph("linkedin.com/in/anthonylpz  ·  github.com/anthonylopez-dev",
    ParagraphStyle('F2', fontSize=9, textColor=C_SECONDARY, fontName='Helvetica',
                   alignment=TA_CENTER, spaceAfter=4)))
elements.append(Paragraph(f"Generado: {datetime.now().strftime('%d de %B de %Y')}",
    ParagraphStyle('F3', fontSize=8, textColor=C_GRAY, fontName='Helvetica-Oblique',
                   alignment=TA_CENTER)))

print("✅ Contenido del PDF en español construido")

✅ Contenido del PDF en español construido


In [19]:
doc.build(elements)
size_mb = output_path.stat().st_size / 1024 / 1024
print(f"✅ PDF generado: {output_path}")
print(f"   Tamaño: {size_mb:.2f} MB")

✅ PDF generado: ..\reports\pdf\techpulse_report_es.pdf
   Tamaño: 2.09 MB
